# LTX-2.5 Pilot — Tek Sahne + Chatterbox Türkçe Anlatım

**Ön koşullar (bir kez, elle):**
1. Runtime → Change runtime type → **GPU (L4 24GB veya A100)**
2. Sol menü → Secrets → **HF_TOKEN** ekle (Hugging Face token'ın)
3. https://huggingface.co/Lightricks/LTX-2.5 adresinde **lisansı kabul et** (gated repo; onay gele kadar model inmez)

Aşağıdaki hücreleri sırayla çalıştır. Tüm mantık repodaki script'lerde — bu defter sadece sarmalayıcı.

In [ ]:
#@title 1. Repoyu çek ve kur (idempotent) { display-mode: "form" }
REPO_URL = "https://github.com/Yuki-TR/mobydick.git"  #@param {type:"string"}
REQUIRED_COMMIT = "482eda53bc4a7560197f4d47d3b946d1fa1b5f5a"  #@param {type:"string"}
import pathlib, subprocess
ROOT = pathlib.Path("/content/mobydick")
if not (ROOT / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
subprocess.run(["git", "fetch", "origin"], cwd=ROOT, check=True)
subprocess.run(["git", "checkout", "--detach", "origin/main"], cwd=ROOT, check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip()
print("checkout:", head)
if REQUIRED_COMMIT and head != REQUIRED_COMMIT:
    raise RuntimeError(f"Beklenen commit {REQUIRED_COMMIT}, gelen {head}")
subprocess.run(["pip", "install", "-q", "-e", str(ROOT)], check=True)
subprocess.run(["pip", "install", "-q", "chatterbox-tts"], check=True)
print("kurulum tamam")

In [ ]:
#@title 2. Pinli LTX-2.5 manifest'i üret (SHA'lar HF'den) { display-mode: "form" }
from google.colab import userdata
import os, subprocess
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
subprocess.run(["python", "scripts/ltx25/fetch_manifest.py", "--output", "manifest.ltx25.json"], cwd=ROOT, check=True)
print(open(ROOT / "manifest.ltx25.json").read()[:600])

In [ ]:
#@title 3. Pilotu çalıştır: kurulum → ComfyUI → Türkçe anlatım → FLF2V render → mux { display-mode: "form" }
DURATION = 14  #@param {type:"integer"}
SEED = 0  #@param {type:"integer"}
import subprocess
cmd = [
    "python", "scripts/ltx25/pilot_run.py",
    "--manifest", "manifest.ltx25.json",
    "--first", "projects/moby_dick_pilot/assets/keyframes/001_street.png",
    "--last", "projects/moby_dick_pilot/assets/keyframes/002_coffin.png",
    "--duration", str(DURATION),
]
if SEED:
    cmd += ["--seed", str(SEED)]
proc = subprocess.run(cmd, cwd=ROOT)
raise SystemExit(proc.returncode)